analysis -> feat eng -> model -> web app

In [ ]:
import pandas as pd  # for loading and manipulating tabular data
import numpy as np  # for numerical operations on arrays
import matplotlib.pyplot as plt  # for creating plots and charts
import seaborn as sns  # for statistical data visualization built on matplotlib

In [ ]:
# show all columns when printing a DataFrame instead of truncating with "..."
pd.set_option("display.max_columns", None)

In [ ]:
# apply a white grid background style to all seaborn plots
sns.set_style("whitegrid")

In [ ]:
# load the German Credit dataset from CSV into a DataFrame
df = pd.read_csv("german_credit_data.csv")

In [ ]:
# preview the first 5 rows of the dataset
df.head()

In [ ]:
# count how many rows fall into each Risk category (good/bad)
df["Risk"].value_counts()

In [ ]:
# check the number of rows and columns in the dataset
df.shape

In [ ]:
# show column names, non-null counts, and dtypes for each column
df.info()

In [ ]:
# compute summary statistics for every column (numeric and categorical) and transpose for readability
df.describe(include="all").T

In [ ]:
# list the distinct values present in the Job column
df["Job"].unique()

In [ ]:
# count missing (NaN) values in each column
df.isna().sum()

In [ ]:
# count fully duplicated rows in the dataset
df.duplicated().sum()

In [ ]:
# drop rows containing any missing value and reset the row index
df = df.dropna().reset_index(drop=True)

In [ ]:
# display the cleaned DataFrame
df

In [ ]:
# list the current column names
df.columns

In [ ]:
# remove the unused index column carried over from the CSV export
df.drop(columns = 'Unnamed: 0' , inplace = True)

In [ ]:
# confirm the column was dropped
df.columns

In [ ]:
# plot histograms for the numerical columns to see their distributions
df[["Age", "Credit amount", "Duration"]].hist(bins = 7, edgecolor = "black")
plt.suptitle("Distribution of numerical features", fontsize = 14)  # add an overall title above the subplots
plt.show()  # render the figure

In [ ]:
plt.figure(figsize= (10,5))  # create a wide figure to hold 3 side-by-side boxplots
for i, col in enumerate (["Age", "Credit amount", "Duration"]):
    plt.subplot(1,3,i+1)  # place each boxplot in its own subplot slot
    sns.boxplot(y = df[col], color = "skyblue")  # boxplot to visualize spread and outliers
    plt.title(col)
plt.tight_layout  # NOTE: missing () so this doesn't actually apply tight layout, it's a no-op
plt.show()

In [ ]:
# inspect rows where loan Duration is 60 months or more (long-duration outliers)
df.query("Duration >= 60")

In [ ]:
# list of categorical feature columns to explore/plot together
categorical_cols = ["Sex","Job","Housing", "Saving accounts", "Checking account", "Purpose"]

In [ ]:
plt.figure(figsize= (10,10))  # large figure to fit a 3x3 grid of countplots
for i, col in enumerate(categorical_cols):
    plt.subplot(3,3,i+1)  # one subplot per categorical column
    sns.countplot(data= df, x = col, palette = "Set2", order = df[col].value_counts().index)  # bar count of each category, ordered by frequency
    plt.title(f"Distribution of {col}")
    
plt.tight_layout()  # adjust spacing so subplot titles/labels don't overlap
plt.show()


In [ ]:
# compute pairwise correlation between numeric columns
corr = df[["Age","Job","Credit amount","Duration"]].corr()

In [ ]:
# display the correlation matrix
corr

In [ ]:
# visualize the correlation matrix as a heatmap with correlation values annotated
sns.heatmap(corr, annot=True, cmap= "coolwarm", fmt= ".2f")

with bigger creadit amounts people are looking for bigger duration. Correlation between ceradit amount and duration

In [ ]:
# average credit amount requested per Job category
df.groupby("Job")["Credit amount"].mean()

In [ ]:
# average credit amount requested per Sex
df.groupby("Sex")["Credit amount"].mean()

In [ ]:
# average credit amount broken down by Housing type and loan Purpose
pd.pivot_table(df, values ="Credit amount", index= "Housing", columns = "Purpose")

In [ ]:
# scatter plot of Age vs Credit amount, colored by Sex and point size scaled by Duration
sns.scatterplot(data = df, x="Age", y= "Credit amount", hue="Sex", size = "Duration", alpha = 0.7, palette = "Set1")
plt.title("Credit Amount  vs Age coloresd by Sex and sized by Duration")

In [ ]:
# violin plot showing the distribution of credit amount for each savings account category
sns.violinplot(data = df, x = "Saving accounts", y = "Credit amount", palette="Pastel1")
plt.title("Credit amount distribution by saving accounts")

we can see that saving account isn't really related with the credit amount

In [ ]:
# percentage share of each Risk class in the dataset
df["Risk"].value_counts(normalize=True)*100

In [ ]:
plt.figure(figsize=(8,4))  # figure to hold 3 side-by-side boxplots
for i, col in enumerate(["Age", "Credit amount", "Duration"]):
    plt.subplot(1,3, i+1)
    sns.boxplot(data = df, x = "Risk", y = col, palette = "Pastel2")  # compare distribution of each numeric feature across Risk classes
    plt.title(f"{col} by Risk")

plt.tight_layout()
plt.show()

In [ ]:
# average Age, Credit amount, and Duration for each Risk class
df.groupby("Risk")[["Age", "Credit amount", "Duration"]].mean()

In [ ]:
# display the categorical columns list again for reference
categorical_cols

In [ ]:
plt.figure(figsize=(10,10))  # large figure for a 3x3 grid of countplots
for i, col in enumerate(categorical_cols):
    plt.subplot(3,3,i+1)
    sns.countplot(data = df, x = col, hue = "Risk", palette = "Set1", order = df[col].value_counts().index)  # split each category's counts by Risk class
    plt.title(f"{col} by Risk")
    plt.xticks(rotation=45)  # rotate x labels so they don't overlap

plt.tight_layout()
plt.show()

In [ ]:
# list all columns before selecting features for modeling
df.columns

In [ ]:
# columns to use as input features for the model
features = ["Age", "Sex", "Job", "Housing", "Saving accounts", "Checking account", "Credit amount", "Duration"]

In [ ]:
# column to predict
target = "Risk"

In [ ]:
# build a modeling DataFrame containing only the selected features and target
df_model = df[features + [target]].copy()

In [ ]:
# preview the modeling DataFrame
df_model.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder  # to convert categorical text values into numeric codes
import joblib  # to save/load fitted encoders and models to disk

In [ ]:
# get categorical (text) columns that need encoding, excluding the target
cat_cols = df_model.select_dtypes(include = "object").columns.drop("Risk")

In [ ]:
# dictionary to keep a reference to each column's fitted LabelEncoder
le_dict ={}

In [ ]:
# show which columns will be label-encoded
cat_cols

In [ ]:
for col in cat_cols:
    le = LabelEncoder()  # create a new encoder for this column
    df_model[col] = le.fit_transform(df_model[col])  # fit on the column's values and replace them with numeric codes
    le_dict[col] =le  # keep the fitted encoder for later use (e.g. inverse_transform)
    joblib.dump(le, f"{col}_encoder.pkl")  # persist the encoder to disk for reuse at inference time

In [ ]:
# separate encoder for the target column
le_target = LabelEncoder()


In [ ]:
# confirm the target column name
target

In [ ]:
# encode the target labels (good/bad) into numeric classes
df_model[target] = le_target.fit_transform(df_model[target])

In [ ]:
# check the encoded class counts (0/1) for the target
df_model[target].value_counts()

In [ ]:
# save the target encoder to disk so predictions can be decoded back to labels later
joblib.dump(le_target, "target_encoder.pkl")

In [ ]:
# preview the fully encoded modeling DataFrame
df_model.head()

In [ ]:
from sklearn.model_selection import train_test_split  # to split data into training and test sets

In [ ]:
# feature matrix: all modeling columns except the target
x = df_model.drop(target, axis = 1)


In [ ]:
# target vector
y = df_model[target]

In [ ]:
# display the feature matrix
x

In [ ]:
# display the target vector
y

In [ ]:
# split into train/test sets (80/20), preserving the class balance of Risk via stratify
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,stratify = y, random_state = 1)

In [ ]:
# number of rows/columns in the training feature set
x_train.shape

In [ ]:
# number of rows/columns in the test feature set
x_test.shape

In [ ]:
from sklearn.tree import DecisionTreeClassifier  # simple tree-based classifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier  # ensemble tree-based classifiers
from xgboost import XGBClassifier  # gradient-boosted tree classifier
from sklearn.metrics import accuracy_score  # to evaluate prediction accuracy
from sklearn.model_selection import GridSearchCV  # to search over hyperparameter combinations with cross-validation

In [ ]:
def train_model(model, param_grid, x_train, x_test, y_train, y_test):
    # search param_grid using 5-fold cross-validation, optimizing for accuracy
    grid = GridSearchCV(model, param_grid, cv = 5, scoring = "accuracy", n_jobs= -1)
    grid.fit(x_train, y_train)  # fit on the training data
    best_model = grid.best_estimator_  # retrieve the best-performing model from the search
    y_pred = best_model.predict(x_test)  # predict on the held-out test set
    acc = accuracy_score(y_test, y_train)  # NOTE: bug - compares y_test to y_train instead of y_pred
    return best_model, acc , grid.best_params_


In [ ]:
# decision tree classifier with class_weight="balanced" to account for class imbalance
dt = DecisionTreeClassifier(random_state = 1, class_weight = "balanced")
# hyperparameter grid to search over for the decision tree
dt_param_grid = {
    "max_depth": [3,5,7,10,None],       # maximum depth of the tree
    "min_samples_split": [2,5,10],      # minimum samples required to split an internal node
    "min_samples_leaf": [1,2,4]         # minimum samples required at a leaf node
}

In [ ]:
# train the decision tree with grid search over dt_param_grid and evaluate on the test set
best_dt, acc_dt, params_dt = train_model(dt, dt_param_grid, x_train,x_test, y_train,y_test)